# Week 5, командный этап: классификация аэроснимков (20 классов) — финальный ноутбук

Ноутбук обучает все модели из отчёта и собирает из лучших ансамбль:

* **MLP** и **CNN с нуля** — обязательное сравнение (правила курса: MLP, CNN и хотя бы одна предобученная сеть);
* **предобученные сети** ResNet-50 / 101 / 152 и Inception v3, в том числе с увеличенным входом;
* **ансамбль** — среднее вероятностей предобученных моделей → `final_submission.csv`.

Каждая предобученная модель проходит два этапа:
1. **holdout** — обучение на `train`, оценка на `val` (честная оценка качества, выбор варианта TTA);
2. **full** — те же настройки, обучение заново на `train + val`, предсказание `test`.

**Как запускать.** *Среда выполнения → Сменить среду выполнения → T4 GPU*, затем *Выполнить все*. Результаты каждой
модели сохраняются на Google Drive (`MyDrive/lab5_runs/<модель>`). Если результаты модели там уже есть, она не
обучается заново (`SKIP_DONE = True`), а её сохранённые результаты показываются.

**Как мы запускали на самом деле.** Обучение всех моделей подряд заняло бы около 5 часов T4, поэтому модели
обучались **параллельно в нескольких аккаунтах Colab этим же кодом** (`RUNS = ["<одна модель>"]`), папки
результатов собраны на одном Drive, и финальный прогон с `SKIP_DONE = True` собрал из них ансамбль. Чтобы
воспроизвести всё с нуля, поставьте `SKIP_DONE = False` (время на T4: mlp ≈ 5 мин, cnn ≈ 6, r50 ≈ 25, inc3 ≈ 30,
r101 ≈ 35, r50_320 ≈ 40, r50_mix ≈ 40, inc3_363 ≈ 55, r152 ≈ 55, r101_320 ≈ 60, inc3_363_mix ≈ 75, r152_mix ≈ 80).

## Почему именно такой подход

Исходная точка — индивидуальный чекпоинт: ResNet-18 (224 px, разморожены layer3 + layer4) с аугментацией
искажениями, val 0.934 (чистые снимки 0.939, искажённые 0.921), Kaggle 0.924.

| что ограничивало | решение | почему должно помочь |
|---|---|---|
| слабая сеть: ResNet-18, 70% на ImageNet | **ResNet-50 / 101 / 152** с весами `IMAGENET1K_V2` (81–82% на ImageNet), **Inception v3** | более сильные признаки — прежде всего на чистых снимках, где ResNet-18 застрял на 0.94 |
| половина сети заморожена | **дообучаем всю сеть**, ранним слоям lr в 10 раз меньше, чем новому классификатору | аэроснимки сильно отличаются от фото ImageNet: полезно подстроить и ранние признаки, но аккуратно |
| снимки ужимали 256 → 224 | **256 px — родной размер** (Inception: 299), затем и **увеличенный вход** (320 / 363 px) | не теряем мелкие детали: корты, разметку, пути у станции |
| val (как и test) на 27% состоит из искажённых снимков, а в train их 4% | **RandomCorruption** + случайные кадрирования и цвет; в финале **обучение на train + val** | 699 val-снимков — единственные размеченные примеры из распределения test, среди них 190 настоящих искажённых |
| лучшую эпоху выбирали по val → оценка оптимистична (0.934 на val против 0.924 на Kaggle) | **фиксированное число эпох**, AdamW, разогрев + косинусное снижение lr, label smoothing | берём модель после последней эпохи; val не участвует в обучении |
| одно предсказание на снимок | **TTA**: среднее по 8 вариантам (4 поворота × отражение) и подбор размера тестового снимка | у снимка сверху нет «верха»; усреднение снижает случайные ошибки |
| одна модель | **ансамбль** разных архитектур | ResNet и Inception ошибаются на разных снимках |
| ансамбль насытился: 7 моделей не лучше 4, модели ошибаются одинаково | второй рецепт: **CutMix / MixUp** и 30 эпох | другие ошибки у моделей → больше пользы от усреднения; сильная регуляризация на 5.7 тыс. снимков |

**Что сознательно НЕ делаем:**
* ничего не обучаем и не настраиваем по `test`: никаких псевдо-меток, никакой подстройки BatchNorm на тестовых
  снимках, никаких статистик по test — правило «test только для предсказаний»;
* никаких внешних данных и сетей вне разрешённого списка (ResNeXt, Wide ResNet, EfficientNet, трансформеры);
* модели и варианты не выбираем по публичному лидерборду: решения принимаются по val, правило отбора моделей
  в ансамбль зафиксировано заранее (раздел 11);
* кросс-валидацию на 5 фолдах не делаем: она в 5 раз дороже, а схема holdout → full даёт те же два результата
  (честную оценку и модель, обученную на всех данных) за два прогона.

## 1. Настройки
* `RUNS` — какие модели обучать (ключи из `PRESETS`); ансамбль в разделе 11 собирается из них.
* `SKIP_DONE` — не обучать заново модели, результаты которых уже есть на Drive.
* `QUICK_TEST = True` — прогон всего ноутбука на крошечной части данных (1 эпоха), только проверка, что код работает;
  результаты пишутся в отдельные папки `<модель>_quick`.
* `DATA_FILE_ID` — id архива с данными на Google Drive: он скачивается, только если архива нет на Drive этого
  аккаунта. Ссылка курса работает около недели, поэтому надёжнее открыть доступ к своей копии архива
  (*Открыть доступ → Все, у кого есть ссылка*) и вставить id из ссылки `https://drive.google.com/file/d/<id>/view`.

Общие настройки (`COMMON`) одинаковы для всех предобученных моделей, чтобы модели различались только архитектурой и
размером входа. MLP и CNN учатся с нуля, поэтому у них свой lr и больше эпох.

`test_offsets` — на сколько пикселей увеличивать тестовый снимок при выборе варианта TTA (раздел 9). Первые четыре
модели проверяли +0, +32 и +64 px; у них точность росла с размером, поэтому для моделей с увеличенным входом
добавлен шаг +96 px. В `PRESETS` записано ровно то, с чем каждая модель реально обучалась.

Модели `*_mix` — второй рецепт: те же сети, но с CutMix / MixUp на половине батчей и 30 эпохами (раздел 8).

In [ ]:
RUNS = ["mlp", "cnn", "r50", "r101", "r152", "inc3", "r50_320", "r101_320", "inc3_363",
        "r50_mix", "r152_mix", "inc3_363_mix"]
MODES = ["holdout", "full"]    # holdout: train -> оценка на val; full: train + val -> предсказания для test
SKIP_DONE = True               # модели, чьи результаты уже лежат на Drive, не обучаем заново
QUICK_TEST = False             # True: 1 эпоха на крошечной части данных — проверка, что всё работает
SEED = 42
DATA_FILE_ID = "1sbnKeVmiOmT4ZZHIbXIxuBWF9CQLWt6s"   # id архива с данными на Google Drive

COMMON = dict(
    lr=2e-4,                 # lr предобученной части (AdamW)
    head_lr_mult=10,         # у нового классификатора lr в 10 раз больше: он учится с нуля
    weight_decay=0.01,       # стандартное значение AdamW
    label_smoothing=0.1,     # правильному классу 0.9 + 0.1/20, остальным по 0.1/20: модель меньше «самоуверенна»
    warmup_epochs=1,         # первую эпоху lr плавно растёт от 0
    crop_scale=(0.5, 1.0),   # RandomResizedCrop: кусок от 50% до 100% площади снимка
    corruption_p=0.5,        # доля train-снимков, которые портим (RandomCorruption)
    jitter=0.3,              # яркость, контраст, насыщенность x0.7...1.3
    dropout=0.2,
    epochs=20,
    batch=32,
    test_offsets=(0, 32, 64),
    mix_p=0.0,               # доля батчей с CutMix / MixUp (раздел 8); 0 — без смешивания
    cutmix_alpha=1.0,        # CutMix: доля вырезаемого куска ~ Beta(1, 1), то есть равномерно от 0 до 1
    mixup_alpha=0.4,         # MixUp: вес второго снимка ~ Beta(0.4, 0.4), чаще всего небольшой
)
SCRATCH = dict(lr=1e-3, head_lr_mult=1, epochs=30, batch=64, test_offsets=(0,), modes=("holdout",))
PRESETS = {
    # обучаются с нуля, только для сравнения: этап full и предсказания test им не нужны
    "mlp":      dict(SCRATCH, arch="mlp", size=32),
    "cnn":      dict(SCRATCH, arch="cnn", size=64, dropout=0.3),
    # предобученные сети на родном размере (Inception: 299 — его стандартный вход)
    "r50":      dict(arch="resnet50", size=256),
    "r101":     dict(arch="resnet101", size=256),
    "r152":     dict(arch="resnet152", size=256),
    "inc3":     dict(arch="inception_v3", size=299),
    # те же сети с увеличенным входом
    "r50_320":  dict(arch="resnet50", size=320, test_offsets=(0, 32, 64, 96)),
    "r101_320": dict(arch="resnet101", size=320, test_offsets=(0, 32, 64, 96)),
    "inc3_363": dict(arch="inception_v3", size=363, test_offsets=(0, 32, 64, 96)),
    # второй рецепт: CutMix / MixUp на половине батчей и 30 эпох вместо 20
    "r50_mix":      dict(arch="resnet50", size=256, epochs=30, mix_p=0.5, test_offsets=(0, 32, 64, 96)),
    "r152_mix":     dict(arch="resnet152", size=256, epochs=30, mix_p=0.5, test_offsets=(0, 32, 64, 96)),
    "inc3_363_mix": dict(arch="inception_v3", size=363, epochs=30, mix_p=0.5, test_offsets=(0, 32, 64, 96)),
}

def get_cfg(run):
    return {**COMMON, **PRESETS[run], "seed": SEED}

def run_name(run):
    return run + ("_quick" if QUICK_TEST else "")

for run in RUNS:
    cfg = get_cfg(run)
    print(f"{run:9s} {cfg['arch']:13s} {cfg['size']} px, {cfg['epochs']} эпох, этапы: {cfg.get('modes', MODES)}")

## 2. Данные и папка для результатов
Google Drive подключается всегда: на нём хранится архив с данными (чтобы не скачивать его каждый раз) и папка
`MyDrive/lab5_runs` с результатами всех моделей. Данные распаковываются в текущую папку Colab, все пути к данным
относительные.

In [ ]:
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")
DRIVE = Path("/content/drive/MyDrive")
ZIP_NAME = "week-5-competition-geo-satellites-classification.zip"
DRIVE_ZIP = DRIVE / ZIP_NAME

if Path("train.csv").exists():
    print("данные уже распакованы")
else:
    if not Path(ZIP_NAME).exists():
        if DRIVE_ZIP.exists():
            !cp "{DRIVE_ZIP}" .
        else:
            !gdown {DATA_FILE_ID} -O {ZIP_NAME}
            !cp {ZIP_NAME} "{DRIVE_ZIP}"
    !unzip -q -o {ZIP_NAME}

DATA_DIR = Path(".")
RESULTS_DIR = DRIVE / "lab5_runs"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
print("результаты:", RESULTS_DIR, "| уже есть:", sorted(p.name for p in RESULTS_DIR.iterdir() if p.is_dir()))

## 3. Библиотеки и воспроизводимость
`seed_everything` фиксирует генераторы случайных чисел Python, NumPy и PyTorch и включает детерминированный режим
cuDNN; перед обучением каждой модели он вызывается заново с её `seed`. Процессы загрузки данных получают свои seed
через `seed_worker` (раздел 6). Полной побитовой воспроизводимости на GPU это не гарантирует (часть операций cuDNN
недетерминирована), но повторный прогон даёт те же результаты с точностью до долей процента.

In [ ]:
import gc
import io
import json
import platform
import random
import time
import zipfile

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageEnhance, ImageFilter
from tqdm.auto import tqdm
from IPython.display import display, Image as ShowImage
import sklearn
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision
from torchvision import transforms, models

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
USE_AMP = DEVICE == "cuda"     # смешанная точность (fp16) на GPU: в 2-3 раза быстрее на T4
NUM_WORKERS = 2                # у Colab 2 ядра CPU: два процесса готовят аугментированные снимки

def seed_everything(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything()
gpu = torch.cuda.get_device_name(0) if DEVICE == "cuda" else "нет (!) — включите T4 GPU"
VERSIONS = {"python": platform.python_version(), "torch": torch.__version__, "torchvision": torchvision.__version__,
            "numpy": np.__version__, "pandas": pd.__version__, "scikit-learn": sklearn.__version__,
            "pillow": Image.__version__, "gpu": gpu}
print(VERSIONS)

## 4. Загрузка снимков в память
Все снимки один раз читаются с диска в массивы `uint8` формы (N, 256, 256, 3): train + val занимают около 1.1 ГБ,
test — 1.6 ГБ (его загружаем, только когда он нужен для предсказаний). Дальше на каждой эпохе снимки берутся из
памяти, без повторного чтения и декодирования JPEG.

In [ ]:
train_table = pd.read_csv(DATA_DIR / "train.csv", dtype={"id": str})
val_table = pd.read_csv(DATA_DIR / "val.csv", dtype={"id": str})
test_table = pd.read_csv(DATA_DIR / "test.csv", dtype={"id": str})
classes = pd.read_csv(DATA_DIR / "classes.csv").sort_values("label").class_name.tolist()
NUM_CLASSES = len(classes)

if QUICK_TEST:   # по 12 train- и 5 val-снимков на класс, 300 test-снимков
    train_table = train_table.groupby("label").head(12).reset_index(drop=True)
    val_table = val_table.groupby("label").head(5).reset_index(drop=True)
    test_table = test_table.head(300).reset_index(drop=True)

def load_images(table, folder):
    images = np.empty((len(table), 256, 256, 3), dtype=np.uint8)
    for i, image_id in enumerate(tqdm(table.id, desc=f"loading {folder}")):
        name = image_id if image_id.endswith(".jpg") else f"{image_id}.jpg"
        with Image.open(DATA_DIR / folder / name) as img:
            img = img.convert("RGB")
            if img.size != (256, 256):
                img = img.resize((256, 256), Image.BICUBIC)
            images[i] = np.asarray(img)
    return images

train_images, train_labels = load_images(train_table, "train"), train_table.label.to_numpy()
val_images, val_labels = load_images(val_table, "val"), val_table.label.to_numpy()

_test_images = None
def get_test_images():
    # test читаем с диска один раз, при первом обращении
    global _test_images
    if _test_images is None:
        _test_images = load_images(test_table, "test")
    return _test_images

print(f"train {train_images.shape} | val {val_images.shape} | test {len(test_table)} снимков | классов {NUM_CLASSES}")

## 5. Какие val-снимки искажены
Тот же анализ, что в индивидуальном чекпоинте: для каждого класса берём 1-й и 99-й перцентили резкости, контраста и
блочности JPEG по **train**-снимкам и отмечаем val-снимки за этими границами, а также почти чёрно-белые. Статистики
считаются только по train и val, test не используется. Отметка нужна, чтобы смотреть точность отдельно на чистых и
на искажённых снимках: в чекпоинте главный резерв качества был именно на искажённых.

In [ ]:
def image_stats(rgb_u8):
    rgb = rgb_u8.astype(np.float32)
    gray = rgb @ np.array([0.299, 0.587, 0.114], dtype=np.float32)
    laplacian = gray[:-2, 1:-1] + gray[2:, 1:-1] + gray[1:-1, :-2] + gray[1:-1, 2:] - 4 * gray[1:-1, 1:-1]
    jumps = np.abs(np.diff(gray, axis=1))
    blockiness = jumps[:, 7::8].mean() / (np.delete(jumps, np.s_[7::8], axis=1).mean() + 1e-9)
    saturation = np.asarray(Image.fromarray(rgb_u8).convert("HSV"), dtype=np.float32)[..., 1].mean()
    return {"sharpness": float(laplacian.var()), "contrast": float(gray.std()),
            "saturation": float(saturation), "blockiness": float(blockiness)}

stats = pd.DataFrame([image_stats(img) for images in (train_images, val_images) for img in tqdm(images, desc="stats")])
stats["label"] = np.concatenate([train_labels, val_labels])
stats["split"] = ["train"] * len(train_images) + ["val"] * len(val_images)

train_stats = stats[stats.split == "train"]
low = train_stats.groupby("label")[["sharpness", "contrast"]].quantile(0.01).add_suffix("_q01")
high = train_stats.groupby("label")[["sharpness", "blockiness"]].quantile(0.99).add_suffix("_q99")
s = stats.join(low, on="label").join(high, on="label")
degraded = ((s.sharpness < s.sharpness_q01) | (s.sharpness > s.sharpness_q99) | (s.blockiness > s.blockiness_q99)
            | (s.saturation < 8) | (s.contrast < s.contrast_q01)).to_numpy()
val_degraded = degraded[stats.split.to_numpy() == "val"]       # в порядке строк val_table
print(f"искажённых val-снимков: {val_degraded.sum()} из {len(val_degraded)} ({val_degraded.mean():.1%}); "
      f"в train: {degraded[stats.split.to_numpy() == 'train'].mean():.1%}")

## 6. Аугментации
Работа делится между CPU и GPU, потому что у Colab всего 2 ядра CPU и они не успевают за T4.

**На CPU** (в двух процессах `DataLoader`), над исходным снимком 256×256:
1. `RandomCorruption` — тот же класс, что в чекпоинте: с вероятностью 0.5 одно или два искажения из найденных в val
   (размытие, низкое разрешение, сильный JPEG, ч/б, шум, низкий контраст). Применяем **первым**, на исходном
   размере: настоящие val/test-снимки тоже были испорчены в 256 px;
2. `RandomResizedCrop` — случайный кусок от 50% до 100% площади, растянутый до размера входа сети: модель учится
   узнавать сцену по её части и при разном масштабе объектов;
3. отражения и повороты на 90° — вместе это все 8 вариантов ориентации снимка (у вида сверху нет «верха»).

**На GPU**, сразу для всего батча: случайные яркость, контраст и насыщенность (×0.7…1.3) и нормализация ImageNet.

Val- и test-снимки не аугментируются: только приведение к размеру входа и нормализация.

In [ ]:
IMAGENET_MEAN = torch.tensor([0.485, 0.456, 0.406], device=DEVICE).view(1, 3, 1, 1)
IMAGENET_STD = torch.tensor([0.229, 0.224, 0.225], device=DEVICE).view(1, 3, 1, 1)
GRAY_WEIGHTS = torch.tensor([0.299, 0.587, 0.114], device=DEVICE).view(1, 3, 1, 1)

class RandomCorruption:
    # С вероятностью p применяет 1 или 2 случайных искажения, похожих на найденные в val-снимках.
    OPS = ("blur", "low_res", "jpeg", "gray", "noise", "low_contrast")

    def __init__(self, p=0.5, img_size=256):
        self.p = p
        self.scale = img_size / 256          # искажения измерялись на снимках 256x256

    def __call__(self, img):
        if random.random() >= self.p:
            return img
        for op in random.sample(self.OPS, k=random.choice([1, 1, 2])):
            img = getattr(self, op)(img)
        return img

    def blur(self, img):                     # гауссово размытие, sigma 0.8...3 px (для 256 px)
        return img.filter(ImageFilter.GaussianBlur(radius=random.uniform(0.8, 3.0) * self.scale))

    def low_res(self, img):                  # уменьшаем в 2...6 раз и увеличиваем обратно: мелкие детали теряются
        size = img.width
        small = max(8, int(size * random.uniform(1 / 6, 1 / 2)))
        return img.resize((small, small), Image.BILINEAR).resize((size, size), Image.BILINEAR)

    def jpeg(self, img):                     # сильное JPEG-сжатие, качество 8...40
        buffer = io.BytesIO()
        img.save(buffer, format="JPEG", quality=random.randint(8, 40))
        buffer.seek(0)
        return Image.open(buffer).convert("RGB")

    def gray(self, img):                     # оттенки серого, 3 канала сохраняются
        return img.convert("L").convert("RGB")

    def noise(self, img):                    # гауссов шум, std 5...20 уровней яркости
        rng = np.random.default_rng(random.getrandbits(32))
        pixels = np.asarray(img, dtype=np.float32)
        pixels = pixels + rng.normal(0, random.uniform(5, 20), pixels.shape)
        return Image.fromarray(np.clip(pixels, 0, 255).astype(np.uint8))

    def low_contrast(self, img):             # контраст x0.4...0.8
        return ImageEnhance.Contrast(img).enhance(random.uniform(0.4, 0.8))

class RandomRot90:
    # поворот на 0, 90, 180 или 270 градусов: точная перестановка пикселей, без интерполяции
    ROTATIONS = (None, Image.Transpose.ROTATE_90, Image.Transpose.ROTATE_180, Image.Transpose.ROTATE_270)

    def __call__(self, img):
        rotation = random.choice(self.ROTATIONS)
        return img if rotation is None else img.transpose(rotation)

class ToUint8Tensor:
    # PIL -> тензор uint8 (3, H, W); перевод в float и нормализация — уже на GPU
    def __call__(self, img):
        return torch.from_numpy(np.array(img, dtype=np.uint8)).permute(2, 0, 1)

def make_train_transform(size, cfg):
    return transforms.Compose([
        RandomCorruption(p=cfg["corruption_p"]),
        transforms.RandomResizedCrop(size, scale=cfg["crop_scale"]),
        transforms.RandomHorizontalFlip(),
        transforms.RandomVerticalFlip(),
        RandomRot90(),
        ToUint8Tensor(),
    ])

def gpu_color_jitter(x, strength, generator):
    # x: батч (B, 3, H, W) со значениями 0...1; у каждого снимка свои случайные множители
    def factor():
        return 1 + (torch.rand(x.shape[0], 1, 1, 1, device=x.device, generator=generator) * 2 - 1) * strength
    x = x * factor()                                                      # яркость
    mean = (x * GRAY_WEIGHTS).sum(1, keepdim=True).mean(dim=(2, 3), keepdim=True)
    x = (x - mean) * factor() + mean                                      # контраст
    gray = (x * GRAY_WEIGHTS).sum(1, keepdim=True)
    x = (x - gray) * factor() + gray                                      # насыщенность
    return x.clamp(0, 1)

def to_model_input(x_u8, size=None, jitter=0.0, generator=None):
    # uint8 (B, 3, H, W) -> нормализованный float-тензор на GPU; size: привести к размеру входа сети
    x = x_u8.to(DEVICE, non_blocking=True).float().div_(255)
    if jitter > 0:
        x = gpu_color_jitter(x, jitter, generator)
    if size is not None and x.shape[-1] != size:
        x = F.interpolate(x, size=(size, size), mode="bilinear", antialias=True, align_corners=False)
    x = (x - IMAGENET_MEAN) / IMAGENET_STD
    return x.contiguous(memory_format=torch.channels_last)

class ArrayDataset(Dataset):
    def __init__(self, images, labels, transform):
        self.images, self.labels, self.transform = images, labels, transform

    def __len__(self):
        return len(self.images)

    def __getitem__(self, i):
        return self.transform(Image.fromarray(self.images[i])), int(self.labels[i])

def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)

def make_train_loader(images, labels, cfg):
    generator = torch.Generator().manual_seed(cfg["seed"])     # фиксированный порядок перемешивания
    return DataLoader(ArrayDataset(images, labels, make_train_transform(cfg["size"], cfg)),
                      batch_size=cfg["batch"], shuffle=True, drop_last=True, num_workers=NUM_WORKERS,
                      worker_init_fn=seed_worker, generator=generator, pin_memory=True,
                      persistent_workers=NUM_WORKERS > 0)

In [ ]:
# превью: два train-снимка и по 7 их случайных вариантов — так их видит модель при обучении (вход 256 px)
random.seed(0); torch.manual_seed(0)
preview_transform = make_train_transform(256, get_cfg("r50"))
preview_generator = torch.Generator(device=DEVICE).manual_seed(0)
fig, axes = plt.subplots(2, 8, figsize=(16, 4.4))
for row, i in enumerate([0, 1]):
    axes[row, 0].imshow(train_images[i]); axes[row, 0].set_title(f"исходный: {classes[train_labels[i]]}", fontsize=8)
    batch = torch.stack([preview_transform(Image.fromarray(train_images[i])) for _ in range(7)])
    batch = gpu_color_jitter(batch.to(DEVICE).float() / 255, COMMON["jitter"], preview_generator)
    for col in range(7):
        axes[row, col + 1].imshow(batch[col].permute(1, 2, 0).cpu().numpy())
        axes[row, col + 1].set_title("аугментация", fontsize=8)
for ax in axes.flat: ax.axis("off")
plt.tight_layout(); plt.show()
seed_everything()

## 7. Модели
Все модели устроены одинаково: «тело», которое превращает снимок в вектор признаков, плюс **наш классификатор**
`Dropout → Linear(→ 20)`, соединённые через `nn.Sequential`.

* **MLP** (вход 32 px, как в бейзлайне курса): снимок вытягивается в вектор 3·32·32 = 3072 чисел, затем
  `Linear 3072→512 → ReLU → Dropout 0.2 → Linear 512→256 → ReLU`. Каждый пиксель для MLP — отдельное число: какие
  пиксели соседние, он не знает.
* **CNN** (вход 64 px, как наша CNN из чекпоинта): 4 блока `(Conv 3×3 → BatchNorm → ReLU) ×2 → MaxPool`
  с 32 / 64 / 128 / 256 каналами и усреднение по пространству. Свёртки ищут один и тот же узор в любом месте снимка.
* **Предобученные сети** из torchvision без своего классификатора на 1000 классов ImageNet (`fc = nn.Identity()`).
  Веса: ResNet — `IMAGENET1K_V2` (80.9 / 81.9 / 82.3% на ImageNet у ResNet-50 / 101 / 152), Inception v3 —
  `IMAGENET1K_V1` (77.3%). Скрытый слой в классификаторе им не нужен: вся сеть дообучается, и её последний блок сам
  подстраивает признаки под наши классы. У Inception v3 отключаем вспомогательный выход (`AuxLogits`, нужен был
  только при обучении на ImageNet) и встроенный dropout (он уже есть в нашем классификаторе). Inception сам
  переводит вход из нормализации ImageNet в свою (`transform_input=True`), поэтому подготовка снимков у всех
  моделей одинаковая.

In [ ]:
BACKBONES = {
    "resnet50": (models.resnet50, models.ResNet50_Weights.IMAGENET1K_V2),
    "resnet101": (models.resnet101, models.ResNet101_Weights.IMAGENET1K_V2),
    "resnet152": (models.resnet152, models.ResNet152_Weights.IMAGENET1K_V2),
    "inception_v3": (models.inception_v3, models.Inception_V3_Weights.IMAGENET1K_V1),
}

def conv_block(c_in, c_out):
    return nn.Sequential(nn.Conv2d(c_in, c_out, 3, padding=1), nn.BatchNorm2d(c_out), nn.ReLU(),
                         nn.Conv2d(c_out, c_out, 3, padding=1), nn.BatchNorm2d(c_out), nn.ReLU(), nn.MaxPool2d(2))

def build_model(cfg):
    arch = cfg["arch"]
    if arch == "mlp":
        body = nn.Sequential(nn.Flatten(), nn.Linear(3 * cfg["size"] ** 2, 512), nn.ReLU(), nn.Dropout(0.2),
                             nn.Linear(512, 256), nn.ReLU())
        n_features = 256
    elif arch == "cnn":
        body = nn.Sequential(conv_block(3, 32), conv_block(32, 64), conv_block(64, 128), conv_block(128, 256),
                             nn.AdaptiveAvgPool2d(1), nn.Flatten())
        n_features = 256
    else:
        constructor, weights = BACKBONES[arch]
        body = constructor(weights=weights)
        if arch == "inception_v3":
            body.aux_logits, body.AuxLogits = False, None
            body.dropout = nn.Identity()
        n_features = body.fc.in_features        # 2048 у всех четырёх сетей
        body.fc = nn.Identity()
    head = nn.Sequential(nn.Dropout(cfg["dropout"]), nn.Linear(n_features, NUM_CLASSES))
    return nn.Sequential(body, head)

for run in RUNS:
    cfg = get_cfg(run)
    print(f"{run:9s} {sum(p.numel() for p in build_model(cfg).parameters()) / 1e6:6.2f} M параметров")

## 8. Обучение и предсказание
**`train_model`**:
* **AdamW** с двумя группами параметров: у предобученной сети lr 2e-4, у нового классификатора — в 10 раз больше
  (он начинает со случайных весов, а сети достаточно небольших поправок). MLP и CNN учатся с нуля, у них одна lr 1e-3;
* **разогрев + косинус**: первую эпоху lr растёт от 0 до максимума (пока классификатор случайный, его большие
  градиенты не должны «сломать» предобученные слои), затем плавно падает до 0 по косинусу. К последней эпохе шаги
  маленькие, модель «оседает» в хорошей точке, поэтому лучшую эпоху выбирать не нужно;
* **label smoothing 0.1**: модель не стремится к уверенности 100%, это смягчает переобучение на 5 тыс. снимков;
* **смешанная точность**: прямой проход в fp16 (тензорные ядра T4), `GradScaler` защищает маленькие градиенты от
  обнуления;
* если передан `val`, после каждой эпохи считаем на нём loss и accuracy — **только для графиков**, на обучение и
  выбор модели это не влияет.

**`mix_batch`** — второй рецепт обучения (модели `*_mix`). Первые семь предобученных моделей ошибались на одних и
тех же снимках, и ансамбль из 7 моделей оказался не лучше ансамбля из 4 ни на val, ни на Kaggle. Нужны были
модели, которые учатся иначе. С вероятностью `mix_p = 0.5` батч смешивается со своей же перестановкой:
* **CutMix** (Yun et al., 2019): прямоугольник из другого снимка вставляется в текущий, метка смешивается
  пропорционально площади. Модель учится узнавать сцену по любой её части и опираться на всю сцену, а не на один
  заметный объект — это прямо про наши путаницы church/palace и lake/wetland;
* **MixUp** (Zhang et al., 2018): два снимка накладываются полупрозрачно с весами `lam` и `1 − lam`, метки
  смешиваются так же. Модель учится давать промежуточные вероятности между классами, вероятности лучше
  откалиброваны, и ансамбль от этого выигрывает.

Loss для смешанного батча: `lam · CE(метка 1) + (1 − lam) · CE(метка 2)`. Это сильная регуляризация, поэтому у
таких моделей 30 эпох вместо 20. Train accuracy у них считается по первой метке и для смешанных батчей занижена.

**`predict_proba`** возвращает вероятности классов. С `tta=True` — среднее по 8 вариантам снимка (4 поворота ×
отражение).

In [ ]:
def mix_batch(x, y, cfg, rng):
    # С вероятностью mix_p смешивает батч с его же перестановкой: CutMix или MixUp (50 / 50).
    # Возвращает новый батч, вторые метки y_b и долю lam первого снимка: loss = lam * CE(y) + (1 - lam) * CE(y_b).
    if rng.random() >= cfg["mix_p"]:
        return x, y, 1.0
    perm = torch.from_numpy(rng.permutation(len(y))).to(x.device)
    if rng.random() < 0.5:                                    # CutMix: вставляем прямоугольник из другого снимка
        H, W = x.shape[-2:]
        cut = np.sqrt(1 - rng.beta(cfg["cutmix_alpha"], cfg["cutmix_alpha"]))   # сторона куска, доля от снимка
        cy, cx = rng.integers(H), rng.integers(W)
        y1, y2 = max(cy - int(H * cut) // 2, 0), min(cy + int(H * cut) // 2, H)
        x1, x2 = max(cx - int(W * cut) // 2, 0), min(cx + int(W * cut) // 2, W)
        x = x.clone()
        x[:, :, y1:y2, x1:x2] = x[perm, :, y1:y2, x1:x2]
        lam = 1 - (y2 - y1) * (x2 - x1) / (H * W)              # доля площади, оставшаяся от первого снимка
    else:                                                     # MixUp: полупрозрачное наложение двух снимков
        lam = float(rng.beta(cfg["mixup_alpha"], cfg["mixup_alpha"]))
        x = lam * x + (1 - lam) * x[perm]
    return x.contiguous(memory_format=torch.channels_last), y[perm], float(lam)

def train_model(model, images, labels, cfg, val=None):
    seed_everything(cfg["seed"])
    model = model.to(DEVICE, memory_format=torch.channels_last)
    loader = make_train_loader(images, labels, cfg)
    epochs = 1 if QUICK_TEST else cfg["epochs"]
    optimizer = torch.optim.AdamW([
        {"params": model[0].parameters(), "lr": cfg["lr"]},
        {"params": model[1].parameters(), "lr": cfg["lr"] * cfg["head_lr_mult"]},
    ], weight_decay=cfg["weight_decay"])
    total_steps, warmup_steps = epochs * len(loader), cfg["warmup_epochs"] * len(loader)

    def lr_factor(step):
        if step < warmup_steps:
            return (step + 1) / warmup_steps
        return 0.5 * (1 + np.cos(np.pi * (step - warmup_steps) / max(1, total_steps - warmup_steps)))

    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_factor)
    criterion = nn.CrossEntropyLoss(label_smoothing=cfg["label_smoothing"])
    scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)
    jitter_generator = torch.Generator(device=DEVICE).manual_seed(cfg["seed"])
    mix_rng = np.random.default_rng(cfg["seed"])

    history = []
    for epoch in range(1, epochs + 1):
        model.train()
        start, total_loss, correct, seen = time.time(), 0.0, 0, 0
        for x_u8, y in loader:
            x = to_model_input(x_u8, jitter=cfg["jitter"], generator=jitter_generator)
            y = y.to(DEVICE, non_blocking=True)
            x, y_b, lam = mix_batch(x, y, cfg, mix_rng)
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=USE_AMP):
                outputs = model(x)
                loss = criterion(outputs, y)
                if lam < 1.0:
                    loss = lam * loss + (1 - lam) * criterion(outputs, y_b)
            optimizer.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            total_loss += loss.item() * len(y)
            correct += (outputs.argmax(1) == y).sum().item()
            seen += len(y)
        record = {"epoch": epoch, "train_loss": total_loss / seen, "train_acc": correct / seen,
                  "minutes": (time.time() - start) / 60}
        if val is not None:
            probs = predict_proba(model, val[0], cfg["size"], tta=False)
            record["val_loss"] = float(-np.log(probs[np.arange(len(val[1])), val[1]] + 1e-12).mean())
            record["val_acc"] = float((probs.argmax(1) == val[1]).mean())
        history.append(record)
        print(" | ".join(f"{k} {v:.4f}" if isinstance(v, float) else f"{k} {v}" for k, v in record.items()))
    return model, pd.DataFrame(history)

@torch.no_grad()
def predict_proba(model, images, size, tta=True, batch=64):
    model.eval()
    result = []
    for start in range(0, len(images), batch):
        x = to_model_input(torch.from_numpy(images[start:start + batch]).permute(0, 3, 1, 2), size)
        views = [x]
        if tta:
            views = [torch.rot90(v, k, dims=(2, 3)).contiguous(memory_format=torch.channels_last)
                     for v in (x, x.flip(3)) for k in range(4)]
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=USE_AMP):
            probs = sum(model(v).float().softmax(1) for v in views) / len(views)
        result.append(probs.cpu())
    return torch.cat(result).numpy()

def accuracy_report(probs, labels, degraded):
    correct = probs.argmax(1) == labels
    acc = correct.mean()
    return {"acc": acc, "se": np.sqrt(acc * (1 - acc) / len(correct)),
            "acc_clean": correct[~degraded].mean(), "acc_degraded": correct[degraded].mean()}

def plot_history(history, title, path):
    fig, axes = plt.subplots(1, 2, figsize=(12, 3.6))
    axes[0].plot(history.epoch, history.train_loss, label="train (с аугментациями)")
    axes[0].plot(history.epoch, history.val_loss, label="val")
    axes[1].plot(history.epoch, history.train_acc, label="train (с аугментациями)")
    axes[1].plot(history.epoch, history.val_acc, label="val (без TTA)")
    axes[0].set_title(f"{title}: loss"); axes[1].set_title(f"{title}: accuracy")
    for ax in axes: ax.set_xlabel("эпоха"); ax.legend()
    plt.tight_layout(); plt.savefig(path, dpi=80); plt.show()

## 9. Один эксперимент: holdout → full
`run_experiment(run)` делает для одной модели всё, что нужно для отчёта и ансамбля.

**holdout** — обучение на train, оценка на val. После обучения сравниваем несколько вариантов предсказания:
* без TTA — одна копия снимка;
* TTA ×8 на размере входа и на увеличенных размерах (`test_offsets`). Зачем увеличивать: при обучении
  `RandomResizedCrop` берёт в среднем ~75% площади и растягивает её, то есть объекты на обучающих снимках выглядят
  крупнее, чем на целом тестовом снимке. Увеличение тестового снимка выравнивает масштаб (эффект FixRes,
  Touvron et al., 2019);
* среднее по всем размерам.

Для test берём вариант с TTA, лучший по val; при равенстве — более дешёвый (он выше в списке). Выбор из 4–5 вариантов
на 699 снимках немного завышает оценку лучшего, но заметно меньше, чем выбор лучшей эпохи.

**full** — та же модель с теми же настройками и тем же числом эпох обучается заново на всех 5698 размеченных
снимках и предсказывает test выбранным вариантом. Оценить её на val уже нельзя (val в обучении); её ожидаемое
качество — оценка holdout, скорее чуть лучше: данных на 14% больше, и среди них 190 настоящих искажённых снимков.
`test` используется только здесь и только для предсказания.

Результаты (`summary.json`, `val_probs.npy`, `test_probs.npy`, история обучения, кривые, `submission_<модель>.csv`)
сохраняются в `lab5_runs/<модель>`. Веса full-модели тоже сохраняются (`model_full_fp16.pt`) — на случай, если
понадобится заново предсказать test.

**Продолжение прерванного прогона.** Бесплатный Colab может оборвать сессию. При `SKIP_DONE = True` уже сделанные
этапы не повторяются: если сохранены результаты holdout, вариант TTA берётся из `val_options.csv`; если сохранены
веса full-модели, они загружаются, и остаётся только предсказать test. Так было с `r152_mix`: сессия оборвалась
во время предсказания test, после 2 × 30 эпох обучения. Веса хранятся в fp16, поэтому предсказания такой модели
могут отличаться от «непрерванных» в единичных снимках.

In [ ]:
def show_saved(run_dir):
    # результаты модели, обученной раньше: таблица вариантов на val и кривые обучения
    if (run_dir / "val_options.csv").exists():
        print(pd.read_csv(run_dir / "val_options.csv", index_col=0).round(4).to_string())
    if (run_dir / "curves_holdout.png").exists():
        display(ShowImage(filename=str(run_dir / "curves_holdout.png")))

def run_experiment(run):
    cfg, name = get_cfg(run), run_name(run)
    run_dir = RESULTS_DIR / name
    if SKIP_DONE and (run_dir / "summary.json").exists():
        print(f"===== {name}: результаты уже есть, обучение пропускаем =====")
        show_saved(run_dir)
        return json.loads((run_dir / "summary.json").read_text(encoding="utf-8"))
    run_dir.mkdir(parents=True, exist_ok=True)
    modes = cfg.get("modes", MODES)
    summary = {"run": name, **{k: list(v) if isinstance(v, tuple) else v for k, v in cfg.items()}}
    test_sizes = [cfg["size"]]
    print(f"===== {name}: {cfg['arch']}, вход {cfg['size']} px, {cfg['epochs']} эпох, этапы {list(modes)} =====")

    holdout_saved = (run_dir / "val_probs.npy").exists() and (run_dir / "val_options.csv").exists()
    if "holdout" in modes and SKIP_DONE and holdout_saved:
        # этап holdout уже прошёл в прерванной сессии: берём его сохранённые результаты
        report = pd.read_csv(run_dir / "val_options.csv", index_col=0)
        best = max([k for k in report.index if "TTA" in k], key=lambda k: report.loc[k, "acc"])
        test_sizes = [int(size) for size in best.split(" px")[0].split("+")]
        summary.update({"test_option": best, "test_sizes": test_sizes, "val_acc_no_tta": float(report.iloc[0]["acc"]),
                        "holdout_train_min": float(pd.read_csv(run_dir / "history_holdout.csv").minutes.sum()),
                        **{f"val_{k}": float(v) for k, v in report.loc[best].items()}})
        print(f"holdout уже был: {best} -> val accuracy {report.loc[best, 'acc']:.4f}")
    elif "holdout" in modes:
        seed_everything(cfg["seed"])
        start = time.time()
        model, history = train_model(build_model(cfg), train_images, train_labels, cfg, val=(val_images, val_labels))
        summary["holdout_train_min"] = (time.time() - start) / 60
        history.to_csv(run_dir / "history_holdout.csv", index=False)
        plot_history(history, name, run_dir / "curves_holdout.png")

        S = cfg["size"]
        sizes = [S + d for d in cfg["test_offsets"]]
        tta_probs = {size: predict_proba(model, val_images, size, tta=True) for size in sizes}
        options = {f"{S} px, без TTA": ([S], predict_proba(model, val_images, S, tta=False))}
        options.update({f"{size} px, TTA x8": ([size], p) for size, p in tta_probs.items()})
        if len(sizes) > 1:
            options[f"{'+'.join(map(str, sizes))} px, TTA x8"] = (sizes, np.mean(list(tta_probs.values()), axis=0))
        report = pd.DataFrame({k: accuracy_report(p, val_labels, val_degraded) for k, (_, p) in options.items()}).T
        print(report.round(4).to_string())
        best = max([k for k in options if "TTA" in k], key=lambda k: report.loc[k, "acc"])
        test_sizes = options[best][0]
        np.save(run_dir / "val_probs.npy", options[best][1])
        report.to_csv(run_dir / "val_options.csv")
        summary.update({"test_option": best, "test_sizes": test_sizes, "val_acc_no_tta": float(report.iloc[0]["acc"]),
                        **{f"val_{k}": float(v) for k, v in report.loc[best].items()}})
        print(f"для test: {best} -> val accuracy {report.loc[best, 'acc']:.4f}")
        del model
        gc.collect(); torch.cuda.empty_cache()

    weights_path = run_dir / "model_full_fp16.pt"
    if "full" in modes and SKIP_DONE and weights_path.exists():
        # full-модель уже обучена в прерванной сессии: загружаем её веса и только предсказываем test
        model = build_model(cfg)
        state = torch.load(weights_path, map_location="cpu")
        model.load_state_dict({k: v.float() if v.is_floating_point() else v for k, v in state.items()})
        model = model.to(DEVICE, memory_format=torch.channels_last)
        summary["full_train_min"] = float(pd.read_csv(run_dir / "history_full.csv").minutes.sum())
        print("full-модель уже обучена: загружены сохранённые веса")
    elif "full" in modes:
        seed_everything(cfg["seed"])
        all_images = np.concatenate([train_images, val_images])
        all_labels = np.concatenate([train_labels, val_labels])
        start = time.time()
        model, history_full = train_model(build_model(cfg), all_images, all_labels, cfg)
        summary["full_train_min"] = (time.time() - start) / 60
        history_full.to_csv(run_dir / "history_full.csv", index=False)
        torch.save({k: v.half() if v.is_floating_point() else v for k, v in model.state_dict().items()},
                   weights_path)
        del all_images
        gc.collect()

    if "full" in modes:
        start = time.time()
        test_probs = np.mean([predict_proba(model, get_test_images(), size, tta=True) for size in test_sizes], axis=0)
        summary["predict_min"] = (time.time() - start) / 60
        np.save(run_dir / "test_probs.npy", test_probs)
        submission = pd.DataFrame({"id": test_table.id, "label": test_probs.argmax(1)})
        assert submission.id.tolist() == test_table.id.tolist() and submission.label.between(0, NUM_CLASSES - 1).all()
        submission.to_csv(run_dir / f"submission_{name}.csv", index=False)
        print(f"test: {len(submission)} предсказаний за {summary['predict_min']:.1f} мин, размеры {test_sizes}")
        del model
        gc.collect(); torch.cuda.empty_cache()

    summary["versions"] = VERSIONS
    (run_dir / "summary.json").write_text(json.dumps(summary, indent=1, ensure_ascii=False, default=float),
                                          encoding="utf-8")
    return summary

## 10. Обучение всех моделей
**Как читать вывод каждой модели.**
* *Кривые:* train-loss выше val-loss — это нормально: обучающие снимки испорчены аугментациями, а loss включает
  label smoothing. Если val-accuracy ещё растёт на последних эпохах, эпох мало; если падает задолго до конца — много.
* *Таблица вариантов:* `acc` ± `se` — точность на val и её стандартная ошибка (≈ 0.007 при 0.96): разница меньше
  1–1.5 п.п. — в пределах шума. `acc_clean` и `acc_degraded` — точность на чистых и на искажённых снимках.

In [ ]:
summaries = {run: run_experiment(run) for run in RUNS}

## 11. Сравнение моделей и ансамбль
В ансамбль идут модели, у которых есть и `val_probs.npy` (holdout-модель — для оценки), и `test_probs.npy`
(full-модель — для test), то есть все предобученные. Вероятности усредняются **с равными весами**: веса можно было
бы подобрать по val, но на 699 снимках это почти наверняка подгонка под шум. Оценка ансамбля на val — среднее
holdout-моделей, то есть тех, что val не видели.

**Какие модели берём** — правило зафиксировано заранее, до результатов: все модели, чья точность на val ниже лучшей
не больше чем на 2 стандартные ошибки (≈ 1.2 п.п.). Явно слабая модель не тянет ансамбль вниз, а перебирать
комбинации и брать лучшую по val не нужно: это была бы та же подгонка под шум.

In [ ]:
ENSEMBLE = None     # None = правило выше; или явный список, например ["r50", "r101", "inc3"]

folder = {run: RESULTS_DIR / run_name(run) for run in RUNS}
candidates = [r for r in RUNS if (folder[r] / "val_probs.npy").exists() and (folder[r] / "test_probs.npy").exists()]
if ENSEMBLE:
    chosen = [r for r in candidates if r in ENSEMBLE]
else:
    best = max(candidates, key=lambda r: summaries[r]["val_acc"])
    threshold = summaries[best]["val_acc"] - 2 * summaries[best]["val_se"]
    chosen = [r for r in candidates if summaries[r]["val_acc"] >= threshold]
    print(f"порог: val acc >= {threshold:.4f}; не вошли: {sorted(set(candidates) - set(chosen)) or 'нет'}")

rows = [{"model": r, "arch": s["arch"], "input px": s["size"], "test": s.get("test_option"), "epochs": s["epochs"],
         "seed": s["seed"], "val acc": s["val_acc"], "± SE": s["val_se"], "val acc без TTA": s["val_acc_no_tta"],
         "clean": s["val_acc_clean"], "degraded": s["val_acc_degraded"],
         "train min (holdout)": s.get("holdout_train_min"), "train min (full)": s.get("full_train_min")}
        for r, s in summaries.items()]
ensemble_val = np.mean([np.load(folder[r] / "val_probs.npy") for r in chosen], axis=0)
ens = accuracy_report(ensemble_val, val_labels, val_degraded)
rows.append({"model": "ансамбль: " + " + ".join(chosen), "val acc": ens["acc"], "± SE": ens["se"],
             "clean": ens["acc_clean"], "degraded": ens["acc_degraded"]})
experiments = pd.DataFrame(rows)
experiments.to_csv("experiments_team.csv", index=False)
experiments.round(4)

In [ ]:
# матрица ошибок ансамбля на val (holdout-модели) и самые частые путаницы
pred = ensemble_val.argmax(1)
fig, ax = plt.subplots(figsize=(11, 10))
ConfusionMatrixDisplay(confusion_matrix(val_labels, pred, labels=range(NUM_CLASSES)), display_labels=classes).plot(
    ax=ax, xticks_rotation=90, colorbar=False, cmap="Blues")
ax.set_title(f"Ансамбль на val: {ens['acc']:.4f} ({(pred != val_labels).sum()} ошибок из {len(val_labels)})")
plt.tight_layout(); plt.show()

wrong = pred != val_labels
confusions = (pd.DataFrame({"true": np.array(classes)[val_labels[wrong]], "predicted": np.array(classes)[pred[wrong]],
                            "degraded": val_degraded[wrong]})
              .groupby(["true", "predicted"]).agg(count=("degraded", "size"), degraded=("degraded", "sum"))
              .sort_values("count", ascending=False))
print(f"ошибок: {wrong.sum()}, из них на искажённых снимках: {val_degraded[wrong].sum()}")
confusions

In [ ]:
# финальные предсказания: среднее вероятностей full-моделей ансамбля
test_probs = np.mean([np.load(folder[r] / "test_probs.npy") for r in chosen], axis=0)
final = pd.DataFrame({"id": test_table.id, "label": test_probs.argmax(1)})
assert list(final.columns) == ["id", "label"]
assert final.id.tolist() == test_table.id.tolist()
assert final.label.between(0, NUM_CLASSES - 1).all()
final_name = "final_submission_quick.csv" if QUICK_TEST else "final_submission.csv"
final.to_csv(final_name, index=False)
shares = final.label.value_counts(normalize=True).sort_index() * 100
print(f"{final_name}: {len(final)} строк, ансамбль из {len(chosen)} моделей; доли классов {shares.min():.1f}...{shares.max():.1f}%")
for r in chosen:
    single = np.load(folder[r] / "test_probs.npy").argmax(1)
    print(f"  совпадение с {r}: {(single == final.label.to_numpy()).mean():.4f}")

## 12. Версии и выгрузка
Всё, кроме весов моделей, упаковывается в `results_all.zip`: папки моделей, `final_submission.csv`,
`experiments_team.csv`.

In [ ]:
print(json.dumps(VERSIONS, indent=1, ensure_ascii=False))
archive = Path("results_all_quick.zip" if QUICK_TEST else "results_all.zip")
with zipfile.ZipFile(archive, "w") as z:
    for run in RUNS:
        for f in sorted(folder[run].iterdir()):
            if f.suffix != ".pt":
                z.write(f, arcname=f"runs/{folder[run].name}/{f.name}")
    for f in (final_name, "experiments_team.csv"):
        z.write(f, arcname=f)
try:
    from google.colab import files
    files.download(str(archive))
except Exception as error:
    print("скачайте архив вручную из панели файлов:", archive, error)